# Parabolic character varieties of knots via quandle representations

Use a **SageMath kernel** and run the notebook from top to bottom. Change the
pretzel below, then use just two functions:

```python
J = character_ideal(pretzel)
I = representation_ideal(pretzel)
```

Both return ordinary Sage ideals. By default they describe the parabolic
quandle $Q_2 = \mathbb{C}^2 / \mathbb{Z}_2$. Use argument `parabolic=False` to compute these ideals for the semiparabolic quandle $\widetilde Q_2 = \mathbb{C}^2$.
No Gröbner basis is computed automatically. Arithmetic is over $\mathbb Q$.

## Reference ##
- Ángel González-Prieto and Javier Martínez. *Parabolic character varieties of knots via quandle representations*.

### Description of the pretzel knot

Each pair `(i, q)` represents the tangle $T_i^q$ from Table 1 in the main reference, which corresponds to the $i$-th type of tangle with $q$ twists.
- $T_1$ corresponds to the directed braid oriented upwards.
- $T_2$ corresponds to the directed braid oriented downwards.
- $T_3$ corresponds to the alternating braid with the left strand directed upwards.
- $T_4$ corresponds to the alternating braid with the left strand directed upwards.

*Example:* In Section 6.4 it is considered the knot $P(-2,-3,1,3)$, which is represented as the list `[(1, -2), (2, -3), (1, 1), (2, 3)]'.

In [11]:
pretzel = [(1, -2), (2, -3), (1, 1), (2, 3)]

## Definitions

The code follows the notation of Section 6.
- $M$ is the matrix obtained from each of the braids of the knot.
- $D_n$ is the closing matrix for a pretzel knot made of $n$ $2$-braids.
- $a = (a_1, \ldots, a_n)$ is a vector with $a_i \in \mathbb{C}^2$.
- $s_{ij} = \det(a_i, a_j)$.
- $r=MD_n a$ is the vector of equations for the tangle.
- $f_{pk}=\langle r_p,a_k\rangle$
- $\Delta=\{(2,3),(4,5),\ldots,(2n-2,2n-1),(1,2n)\}$ is the collection of pairs to be identified.

If we denote $S=(s_{ij})$, then the matrix $(f_{pk})$ is $MD_nS$.
Indices in `s` and `Delta` start at **1**, as in the paper. Internally, Sage
matrix positions start at 0.

In [12]:
"""Pretzel ideals in SageMath, following Section 6 of the accompanying paper. Auxiliary functions."""

from functools import cache
from itertools import combinations, combinations_with_replacement, permutations
from operator import index
from sage.all import QQ, PolynomialRing, block_diagonal_matrix, identity_matrix, matrix, zero_matrix


def _ring(names):
    """Use a multivariate QQ ring, including the one-variable case n=2."""
    names = tuple(names)
    return PolynomialRing(QQ, len(names), names=names, order="degrevlex")


def _name(prefix, indices, n):
    """Keep names such as s12; use s1_12 when n >= 10."""
    return prefix + ("" if n < 10 else "_").join(map(str, indices))


def twist_matrix(kind, q, s):
    """Matrix of T_kind^q from Table 1; q may be positive, negative, or zero."""
    kind, q = index(kind), index(q)
    if kind not in (1, 2, 3, 4):
        raise ValueError("The tangle type must be 1, 2, 3, or 4.")
    s = -s if kind in (2, 4) else s
    R = s.parent()

    def crossing(x):
        return matrix(R, [[x, 1], [1, 0]] if q > 0 else [[0, 1], [1, x]])

    if kind in (3, 4):
        return crossing(s) ** abs(q)
    P = identity_matrix(R, 2)
    for j in range(abs(q)):
        P = crossing((-1)**j * s) * P
    return P


def pretzel_matrices(pretzel):
    """Return M, D_n, S, s, Delta in a flat dictionary (Section 6.3.1).

    Input: [(type_1, q_1), ..., (type_n, q_n)], with n >= 2.
    M is the block twist matrix, D_n is the opening matrix, and S[i-1,j-1]
    is s_ij=det(a_i,a_j). Delta uses the paper's one-based row labels.
    Thus r=M*D_n*a and the matrix of f_pk=det(r_p,a_k) is M*D_n*S.
    """
    pretzel = tuple(pretzel)
    n = len(pretzel)
    if n < 2:
        raise ValueError("At least two twist blocks are required.")
    edges = list(combinations(range(1, n + 1), 2))
    R = _ring(_name("s", edge, n) for edge in edges)
    s = dict(zip(edges, R.gens()))
    S = zero_matrix(R, n, n)
    for (i, j), value in s.items():
        S[i - 1, j - 1], S[j - 1, i - 1] = value, -value
    D_n = zero_matrix(R, 2*n, n)
    columns = [0] + [j for j in range(1, n) for _ in range(2)] + [0]
    for row, column in enumerate(columns):
        D_n[row, column] = 1
    parameters = [s[i, i + 1] for i in range(1, n)] + [-s[1, n]]
    M = block_diagonal_matrix([
        twist_matrix(kind, q, value)
        for (kind, q), value in zip(pretzel, parameters)
    ])
    Delta = [(2*i, 2*i + 1) for i in range(1, n)] + [(1, 2*n)]
    return {"M": M, "D_n": D_n, "S": S, "s": s, "Delta": Delta}


def invariant_ring(n):
    """Return P,u,t with u_ij=s_ij^2 and t_ijk=s_ij*s_jk*s_ki (Section 6.3.2)."""
    edges = list(combinations(range(1, n + 1), 2))
    triples = list(combinations(range(1, n + 1), 3))
    P = _ring([_name("u", edge, n) for edge in edges]
              + [_name("t", triple, n) for triple in triples])
    u = dict(zip(edges, P.gens()[:len(edges)]))
    t = dict(zip(triples, P.gens()[len(edges):]))
    return P, u, t


def universal_ideal(P, u, t, n):
    """Return H_n from Remark 5.8, as used in Corollary 6.16."""
    def U(i, j):
        return P(0) if i == j else u[tuple(sorted((i, j)))]

    relations = []
    for I, J in combinations_with_replacement(t, 2):
        relations.append(2*t[I]*t[J] - matrix(P, [[U(i, j) for j in J] for i in I]).det())
    for I in combinations(range(1, n + 1), 4):
        for a in range(1, n + 1):
            relations.append(sum((-1)**r * U(a, I[r]) * t[I[:r] + I[r + 1:]]
                                 for r in range(4)))
    return P.ideal([h for h in relations if h != 0])

In [13]:
"""Convertion utilities into other coordinates. Auxiliary functions."""

def invariant_converter(R, P, s, u, t):
    """Return a map from sign-invariant s-polynomials to u,t representatives.

    Squares become u coordinates. Remaining edges form even graphs, which
    split into Euler circuits. The trace identity reduces each circuit to
    u,t. Equality holds modulo the Plucker relations, not in the free s-ring.
    With the paper's cyclic convention, tr(N_i*N_j)=-u_ij and
    tr(N_i*N_j*N_k)=t_ijk for i<j<k, where N_i(v)=a_i*det(a_i,v).
    """
    edges = tuple(s)

    @cache
    def trace(vertices):
        if len(vertices) == 0:
            return P(2)
        if len(vertices) == 1:
            return P(0)
        if len(vertices) == 2:
            a, b = vertices
            return P(0) if a == b else -u[tuple(sorted(vertices))]
        if len(vertices) == 3:
            if len(set(vertices)) < 3:
                return P(0)
            inversions = sum(vertices[i] > vertices[j] for i in range(3) for j in range(i + 1, 3))
            return (-1)**inversions * t[tuple(sorted(vertices))]
        a, b, c = vertices[:3]
        rest = vertices[3:]
        return (trace((a, b))*trace((c,) + rest)
                - trace((a, c))*trace((b,) + rest)
                + trace((b, c))*trace((a,) + rest)
                + trace((a, b, c))*trace(rest)) / 2

    @cache
    def even_graph(odd_edges):
        remaining = set(odd_edges)
        vertices = {v for edge in remaining for v in edge}
        if any(sum(v in edge for edge in remaining) % 2 for v in vertices):
            raise ValueError("The polynomial is not invariant under individual sign changes.")
        value = P(1)
        while remaining:
            stack, circuit, sign = [min(min(e) for e in remaining)], [], 1
            while stack:
                a = stack[-1]
                edge = min((e for e in remaining if a in e), default=None)
                if edge is None:
                    circuit.append(stack.pop())
                else:
                    remaining.remove(edge)
                    b = edge[1] if a == edge[0] else edge[0]
                    sign *= -1 if a > b else 1
                    stack.append(b)
            circuit.reverse()
            value *= sign * trace(tuple(circuit[:-1]))
        return value

    @cache
    def monomial(exponents):
        value, odd_edges = P(1), []
        for edge, exponent in zip(edges, exponents):
            value *= u[edge] ** (exponent // 2)
            if exponent % 2:
                odd_edges.append(edge)
        return value * even_graph(tuple(odd_edges))

    def convert(polynomial):
        return P(sum(coefficient * monomial(tuple(exponents))
                     for exponents, coefficient in R(polynomial).dict().items()))

    return convert


def path_relations(f, s, Delta, convert):
    """Computes the generators of J_K from Corollary 6.16 in u,t coordinates.

    Delta is one-based. For k=l the multiplier is 1; otherwise enumerate
    all simple paths gamma from k to l, with signed factors s_ab=-s_ba.
    """
    n, R = f.ncols(), f.base_ring()
    for p, q in Delta:
        for k, l in combinations_with_replacement(range(1, n + 1), 2):
            g = f[p - 1, k - 1]*f[p - 1, l - 1] - f[q - 1, k - 1]*f[q - 1, l - 1]
            if g == 0:
                continue
            if k == l:
                yield convert(g)
                continue
            interior = [v for v in range(1, n + 1) if v not in (k, l)]
            for length in range(len(interior) + 1):
                for middle in permutations(interior, length):
                    gamma = (k,) + middle + (l,)
                    s_gamma = R(1)
                    for a, b in zip(gamma, gamma[1:]):
                        s_gamma *= s[a, b] if a < b else -s[b, a]
                    yield convert(s_gamma * g)

In [14]:
def character_ideal(pretzel, parabolic=True):
    """Return a Sage ideal defining the character variety.

    parabolic=True: H_n + J_K in QQ[u,t], as in Corollary 6.16.
    parabolic=False: Plucker relations + J_tilde_K in QQ[s], as in Corollary 6.12.
    No Groebner basis is computed automatically.
    """
    data = pretzel_matrices(pretzel)
    M, D_n, S = data["M"], data["D_n"], data["S"]
    s, Delta = data["s"], data["Delta"]
    n, R = D_n.ncols(), M.base_ring()
    f = M * D_n * S

    # Semi-parabolic quandle
    if not parabolic:
        plucker = [s[i, j]*s[k, l] - s[i, k]*s[j, l] + s[i, l]*s[j, k]
                   for i, j, k, l in combinations(range(1, n + 1), 4)]
        J_tilde_K = [f[p - 1, k] - f[q - 1, k] for p, q in Delta for k in range(n)]
        return R.ideal(plucker + J_tilde_K)

    # Parabolic quandle
    P, u, t = invariant_ring(n)
    convert = invariant_converter(R, P, s, u, t)
    H_n = universal_ideal(P, u, t, n)
    
    generators = dict.fromkeys(g for g in path_relations(f, s, Delta, convert) if g != 0)
    J_K = P.ideal(list(generators))
    
    return H_n + J_K


def representation_ideal(pretzel, parabolic=True):
    """Return a Sage ideal defining the representation variety.

    parabolic=True: cone equations + I_K in QQ[x1,y1,z1,...] (Proposition 6.14).
    parabolic=False: I_tilde_K in QQ[a1_1,a1_2,...] (Proposition 6.9).
    Here a{k}_1 and a{k}_2 represent the two components of a_k.
    """
    data = pretzel_matrices(pretzel)
    M, D_n, s, Delta = data["M"], data["D_n"], data["s"], data["Delta"]
    n, R = D_n.ncols(), M.base_ring()
    A = _ring(f"a{k}_{j}" for k in range(1, n + 1) for j in (1, 2))
    a = matrix(A, n, 2, A.gens())
    
    determinants = [a[i - 1, 0]*a[j - 1, 1] - a[i - 1, 1]*a[j - 1, 0] for i, j in s]
    substitute = R.hom(determinants, A)
    coefficients = matrix(A, 2*n, n, [substitute(c) for c in (M * D_n).list()])
    r = coefficients * a
    
    if not parabolic:
        return A.ideal([r[p - 1, j] - r[q - 1, j] for p, q in Delta for j in (0, 1)])
        
    P = _ring(f"{v}{k}" for k in range(1, n + 1) for v in ("x", "y", "z"))
    xyz = [P.gens()[3*k:3*k + 3] for k in range(n)]

    def to_xyz(polynomial):
        value = P(0)
        for exponents, coefficient in A(polynomial).dict().items():
            term = P(coefficient)
            
            for k, (x, y, z) in enumerate(xyz):
                b, c = exponents[2*k:2*k + 2]
                if (b + c) % 2:
                    raise ValueError(f"The equation is not invariant under a_{k + 1} -> -a_{k + 1}.")
                term *= x**(b // 2) * y**(b % 2) * z**(c // 2)
            value += term
            
        return value

    cones = [x*z - y**2 for x, y, z in xyz]
    I_K = [to_xyz(r[p - 1, i]*r[p - 1, j] - r[q - 1, i]*r[q - 1, j])
           for p, q in Delta for i, j in ((0, 0), (0, 1), (1, 1))]
    return P.ideal(cones + I_K)

## Character variety

- If `parabolic = True`: Returns the ideal $J = H_n+J_K$ of Corollary 6.16 in $\mathbb Q[u,t]$.
The invariant coordinates use the convention from Section 6.3.2:
$$u_{ij}=s_{ij}^2,\qquad t_{ijk}=s_{ij}s_{jk}s_{ki}.$$
All simple paths from Remark 6.15 are included.

- If `parabolic = False`: Returns the ideal $\widetilde{J} = \langle f_{pk} - f_{qk}\rangle$ of Proposition 6.10 in $\mathbb Q[s]$.

In [17]:
J = character_ideal(pretzel, parabolic = True)
J

Ideal (-2*u12*u13*u23 + 2*t123^2, -u12*u14*u23 - u12*u13*u24 + u12^2*u34 + 2*t123*t124, u13*u14*u23 - u13^2*u24 + u12*u13*u34 + 2*t123*t134, u14*u23^2 - u13*u23*u24 - u12*u23*u34 + 2*t123*t234, -2*u12*u14*u24 + 2*t124^2, u14^2*u23 - u13*u14*u24 - u12*u14*u34 + 2*t124*t134, u14*u23*u24 - u13*u24^2 + u12*u24*u34 + 2*t124*t234, -2*u13*u14*u34 + 2*t134^2, -u14*u23*u34 - u13*u24*u34 + u12*u34^2 + 2*t134*t234, -2*u23*u24*u34 + 2*t234^2, -u14*t123 + u13*t124 - u12*t134, -u24*t123 + u23*t124 + u12*t234, -u34*t123 - u23*t134 + u13*t234, u34*t124 - u24*t134 + u14*t234, u12^3 - u13*u23^2 - 2*u12^2 - u12*u23 + 2*u13*u23 - 2*u23*t123 + u12 - u13 + 2*t123, -u12^3 + u12*u23^2 + u23^2*t123 + u12^2 - u12*u23 - 2*u23*t123 + t123, u13*u23^3 - 2*u13*u23^2 - u12^2*t123 + u23^2*t123 + u13*u23 + u12*t123 - u23*t123, 1/2*u14*u23^3 + 1/2*u13*u23^2*u24 - 1/2*u12*u23^2*u34 - u14*u23^2 - u13*u23*u24 + u12*u23*u34 - u12^2*t124 + u23^2*t124 + 1/2*u14*u23 + 1/2*u13*u24 - 1/2*u12*u34 + u12*t124 - u23*t124, -1/2*u12^2

In [18]:
# The resulting ideal can be simplified using a Grobner basis algorithm

GJ = J.groebner_basis()
print('Groebner basis:')
print(list(GJ))

d = J.dimension()
print("Dimension:", d)

if d == 0:
    print("Length:", J.vector_space_dimension())

Groebner basis:
[u12^2 - u34, u12*u24 - u12 + u34 - 4*t124 - t134 - 3*t234, u24^2 - u12 - u24 + 4*u34 - 10*t124 - 3*t134 - 8*t234, u12*u34 + 2*u12 + u34 + 4*t234, u24*u34 + 4*u12 - 4*u34 + 12*t124 + 3*t134 + 13*t234, u34^2 - 6*u12 - 11*u34 + 24*t124 + 8*t134, u12*t124 + u12 - u34 + 4*t124 + t134 + 4*t234, u24*t124 + 2*u12 - 4*u34 + 11*t124 + 3*t134 + 9*t234, u34*t124 - 5*u12 + u34 - 6*t124 - t134 - 12*t234, t124^2 - 3*u12 + 4*u34 - 12*t124 - 3*t134 - 11*t234, u12*t134 - u12 - 7*u34 + 14*t124 + 5*t134 + 4*t234, u24*t134 - 6*u12 - 4*t124 - 13*t234, u34*t134 + u12 + 43*u34 - 102*t124 - 31*t134 - 44*t234, t124*t134 + 6*u12 + 9*u34 - 18*t124 - 7*t134 + 3*t234, t134^2 + 16*u12 - 107*u34 + 272*t124 + 81*t134 + 153*t234, u12*t234 + u12 + 3*u34 - 6*t124 - 2*t134 - t234, u24*t234 + 3*u12 + 2*t124 + 6*t234, u34*t234 - u12 - 19*u34 + 46*t124 + 14*t134 + 19*t234, t124*t234 - 3*u12 - 4*u34 + 8*t124 + 3*t134 - 2*t234, t134*t234 - 7*u12 + 50*u34 - 126*t124 - 37*t134 - 70*t234, t234^2 + 3*u12 - 23*u34 

## Representation variety

- If `parabolic = True`: Returns the ideal $I = (x_kz_k-y_k^2)+I_K$ in $\mathbb Q[x,y,z]$ from Proposition 6.14, with
$x_k=(a_k^1)^2$, $y_k=a_k^1a_k^2$, $z_k=(a_k^2)^2$.

- If `parabolic = False`: Returns the ideal $\widetilde{I} = \langle r_{p} - r_{q}\rangle$ of Theorem 6.8 in $\mathbb Q[a]$.

These calculations are independent of the character calculation above.

In [9]:
I = representation_ideal(pretzel, parabolic = False)
I

Ideal (-a1_2^2*a2_1^3 + 2*a1_1*a1_2*a2_1^2*a2_2 - a1_1^2*a2_1*a2_2^2 + a2_2^2*a3_1^3 - 2*a2_1*a2_2*a3_1^2*a3_2 + a2_1^2*a3_1*a3_2^2 + a1_1*a1_2*a2_1 - a1_1^2*a2_2 + a2_1*a2_2*a3_1 - a2_1^2*a3_2 + a2_1 - a3_1, -a1_2^2*a2_1^2*a2_2 + 2*a1_1*a1_2*a2_1*a2_2^2 - a1_1^2*a2_2^3 + a2_2^2*a3_1^2*a3_2 - 2*a2_1*a2_2*a3_1*a3_2^2 + a2_1^2*a3_2^3 + a1_2^2*a2_1 - a1_1*a1_2*a2_2 + a2_2^2*a3_1 - a2_1*a2_2*a3_2 + a2_2 - a3_2, -a2_2^3*a3_1^4 + 3*a2_1*a2_2^2*a3_1^3*a3_2 - 3*a2_1^2*a2_2*a3_1^2*a3_2^2 + a2_1^3*a3_1*a3_2^3 - a2_1*a2_2^2*a3_1^2 + 2*a2_1^2*a2_2*a3_1*a3_2 - a2_1^3*a3_2^2 + 2*a2_2*a3_1^2 - 2*a2_1*a3_1*a3_2 + a3_1*a3_2*a4_1 - a3_1^2*a4_2 + a2_1 - a4_1, -a2_2^3*a3_1^3*a3_2 + 3*a2_1*a2_2^2*a3_1^2*a3_2^2 - 3*a2_1^2*a2_2*a3_1*a3_2^3 + a2_1^3*a3_2^4 - a2_2^3*a3_1^2 + 2*a2_1*a2_2^2*a3_1*a3_2 - a2_1^2*a2_2*a3_2^2 + 2*a2_2*a3_1*a3_2 - 2*a2_1*a3_2^2 + a3_2^2*a4_1 - a3_1*a3_2*a4_2 + a2_2 - a4_2, -a1_2^3*a4_1^4 + 3*a1_1*a1_2^2*a4_1^3*a4_2 - 3*a1_1^2*a1_2*a4_1^2*a4_2^2 + a1_1^3*a4_1*a4_2^3 + a1_1*a1_2^2*a4_1^

In [10]:
# The resulting ideal can be simplified using a Grobner basis algorithm

GI = I.groebner_basis()
print('Groebner basis:')
print(list(GI))

d = I.dimension()
print("Dimension:", d)

if d == 0:
    print("Length:", I.vector_space_dimension())

Groebner basis:
[a2_1*a2_2*a4_1 - a2_1^2*a4_2 - a2_1 + 1/2*a3_1 + 1/2*a4_1, a2_2^2*a4_1 - a2_1*a2_2*a4_2 - a2_2 + 1/2*a3_2 + 1/2*a4_2, a2_2*a4_1^2 - a2_1*a4_1*a4_2 + 1/2*a3_1 - 1/2*a4_1, a2_2*a4_1*a4_2 - a2_1*a4_2^2 + 1/2*a3_2 - 1/2*a4_2, a2_1*a3_1 + a2_1*a4_1 - a3_1*a4_1 - a4_1^2, a2_2*a3_1 + a2_2*a4_1 - a3_1*a4_2 - a4_1*a4_2, a3_1^2 - a4_1^2, a2_1*a3_2 + a2_1*a4_2 - a3_1*a4_2 - a4_1*a4_2, a2_2*a3_2 + a2_2*a4_2 - a3_2*a4_2 - a4_2^2, a3_1*a3_2 - a4_1*a4_2, a3_2^2 - a4_2^2, a3_2*a4_1 - a3_1*a4_2, a1_1 - a2_1, a1_2 - a2_2]
Dimension: 3


## Further calculations



In [21]:
# The matrices M and D_n corresponding to the basic span of the underlying braids can be computed as follows.
data = pretzel_matrices(pretzel)
M, D_n = data["M"], data["D_n"]

show(M)
show(D_n)

[             1            s12|             0              0|             0              0|             0              0]
[          -s12     -s12^2 + 1|             0              0|             0              0|             0              0]
[-----------------------------+-----------------------------+-----------------------------+-----------------------------]
[             0              0|           s23     -s23^2 + 1|             0              0|             0              0]
[             0              0|    -s23^2 + 1  s23^3 - 2*s23|             0              0|             0              0]
[-----------------------------+-----------------------------+-----------------------------+-----------------------------]
[             0              0|             0              0|           s34              1|             0              0]
[             0              0|             0              0|             1              0|             0              0]
[-----------------------------+-----------------------------+-----------------------------+-----------------------------]
[             0              0|             0              0|             0              0|-s14^3 + 2*s14     -s14^2 + 1]
[             0              0|             0              0|             0              0|    -s14^2 + 1           -s14]

[1 0 0 0]
[0 1 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 1 0]
[0 0 0 1]
[0 0 0 1]
[1 0 0 0]

In [24]:
# The ideal H_n described in Remark 5.8 can be computed as follows.
n = len(pretzel)
P, u, t = invariant_ring(n)
H_n = universal_ideal(P, u, t, n)

show(H_n)

Ideal (-2*u12*u13*u23 + 2*t123^2, -u12*u14*u23 - u12*u13*u24 + u12^2*u34 + 2*t123*t124, u13*u14*u23 - u13^2*u24 + u12*u13*u34 + 2*t123*t134, u14*u23^2 - u13*u23*u24 - u12*u23*u34 + 2*t123*t234, -2*u12*u14*u24 + 2*t124^2, u14^2*u23 - u13*u14*u24 - u12*u14*u34 + 2*t124*t134, u14*u23*u24 - u13*u24^2 + u12*u24*u34 + 2*t124*t234, -2*u13*u14*u34 + 2*t134^2, -u14*u23*u34 - u13*u24*u34 + u12*u34^2 + 2*t134*t234, -2*u23*u24*u34 + 2*t234^2, -u14*t123 + u13*t124 - u12*t134, -u24*t123 + u23*t124 + u12*t234, -u34*t123 - u23*t134 + u13*t234, u34*t124 - u24*t134 + u14*t234) of Multivariate Polynomial Ring in u12, u13, u14, u23, u24, u34, t123, t124, t134, t234 over Rational Field